# BFG — Data acquisition

bioRxiv distributes each preprint as a `.meca` archive: a ZIP file containing JATS XML
and the original figure images (PNG/TIFF). The project corpus is the full May 2026 monthly
dump (7,132 articles, ~75 GB compressed).

Two acquisition methods are covered here:

| Method | When to use |
|--------|-------------|
| **A — rclone from S3** | Bulk download of a whole month (~7,000 articles) |
| **B — bioRxiv API** | Download individual articles by DOI |

Method A gives you `.meca` archives with the original-resolution figures. Method B gives you
JATS XML + rendered JPEG thumbnails, which are lower resolution but sufficient for quick testing.

Both methods write under `data/` at the repository root, which is git-ignored. The runbook (`00_run_all.ipynb`, §5 and §9) and the `bfg annotate` CLI expect the `.meca` archives in `data/May_2026/`.

## A — Bulk download via rclone from bioRxiv S3

bioRxiv hosts monthly MECA dumps in the public AWS S3 bucket `s3://biorxiv-src-monthly/`. The bucket is **requester-pays**: your AWS account is charged for the egress (roughly \$7–10 for a full month at standard S3 rates). No IAM policy is required beyond a standard AWS account.

Recent months sit in `Current_Content/<Month>_<Year>/` (for example `Current_Content/May_2026/`); older content is moved into batches under `Back_Content/`. The listing below shows which folders exist.

In [ ]:
# Verify rclone is installed
import shutil, subprocess
rclone = shutil.which("rclone")
print(f"rclone: {rclone or 'NOT FOUND'}")
if rclone:
    r = subprocess.run([rclone, "--version"], capture_output=True, text=True)
    print(r.stdout.split("\n")[0])

### Configure rclone

Create a remote called `biorxiv` pointing at S3. Run this once in a terminal:

```bash
rclone config create biorxiv s3 \
  provider=AWS \
  region=us-east-1 \
  requester_pays=true \
  access_key_id=YOUR_AWS_KEY \
  secret_access_key=YOUR_AWS_SECRET
```

If you are on a corporate network (Zscaler / custom CA), add:

```bash
  ca_cert=/path/to/cacert_with_zscaler.pem
```

**AWS credentials are required.** Requester-pays buckets reject anonymous requests, so the access key must belong to an AWS account with billing enabled; the transfer is charged to that account.

### List available months

Before downloading, confirm the path for the month you want.

In [ ]:
import subprocess, shutil

rclone = shutil.which("rclone")

# rclone S3 paths are remote:bucket/prefix
BUCKET_ROOT = "biorxiv:biorxiv-src-monthly/Current_Content"

if rclone:
    r = subprocess.run(
        [rclone, "lsd", BUCKET_ROOT, "--s3-requester-pays"],
        capture_output=True, text=True
    )
    print(r.stdout or "(no output)")
    if r.stderr:
        print("STDERR:", r.stderr[:400])
else:
    print("rclone not on PATH — install from https://rclone.org/downloads/")

In [ ]:
# Download one month
# Edit MONTH to match the folder name shown by the listing above.

from pathlib import Path

MONTH = "May_2026"                 # folder under Current_Content/
DEST  = Path("../data") / MONTH    # data/May_2026/ at the repository root
DEST.mkdir(parents=True, exist_ok=True)

S3_PATH = f"{BUCKET_ROOT}/{MONTH}/"

rclone_cmd = [
    rclone or "rclone",
    "copy", S3_PATH, str(DEST),
    "--s3-requester-pays",
    "--transfers", "8",
    "--progress",
    "--stats-one-line",
]

print("Command:")
print("  " + " ".join(rclone_cmd))
print(f"\nDestination: {DEST.resolve()}")
print("\nRun the cell below to start. Expect ~75 GB and 30–90 min on a 100 Mbps link.")

In [ ]:
# Uncomment to run. Large download — do NOT run in a shared CI environment.
#
# import subprocess
# r = subprocess.run(rclone_cmd, text=True)
# print(f"exit code: {r.returncode}")

## B — Per-article download via bioRxiv API

For testing or fetching a handful of articles by DOI. Downloads JATS XML + rendered JPEG figures (not the original .meca). Slower per-article and lower-resolution images, but works without AWS credentials.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
from datetime import date, timedelta
import re, time, xml.etree.ElementTree as ET
import httpx

# Target DOIs — edit this list
DOIS = [
    "10.1101/2024.01.24.577109",   # worked example from the BFG demo
]

SERVER = "biorxiv"         # "biorxiv" or "medrxiv"
OUT_DIR = Path("../data/biorxiv_api")   # data/biorxiv_api/ at the repository root
IMAGE_VARIANT = "large"    # "large" | "medium" | "small"

REQUEST_DELAY_S = 1.0
TIMEOUT_S = 60.0
USER_AGENT = "cm3070-bfg/0.1 (biomedical figure research)"

_root = Path.cwd()
while _root.name and not (_root / "pyproject.toml").exists():
    _root = _root.parent
_bundle = _root / "cacert_with_zscaler.pem"
CA_BUNDLE = str(_bundle) if _bundle.exists() else True

_client = httpx.Client(
    headers={"User-Agent": USER_AGENT, "Accept": "*/*"},
    verify=CA_BUNDLE,
    timeout=TIMEOUT_S,
    follow_redirects=True,
)

def polite_get(url: str) -> httpx.Response:
    time.sleep(REQUEST_DELAY_S)
    r = _client.get(url)
    r.raise_for_status()
    return r

print(f"CA bundle: {CA_BUNDLE}")
print(f"Output:    {OUT_DIR.resolve()}")

In [ ]:
API_BASE = "https://api.biorxiv.org"
_NS_XLINK = "http://www.w3.org/1999/xlink"

def fetch_by_doi(server: str, doi: str) -> list[dict]:
    url = f"{API_BASE}/details/{server}/{doi}"
    return polite_get(url).json().get("collection", [])

def extract_figures(xml_bytes: bytes) -> list[dict]:
    root = ET.fromstring(xml_bytes)
    figs: list[dict] = []
    for fig in root.iter("fig"):
        g = fig.find(".//graphic")
        href = g.get(f"{{{_NS_XLINK}}}href") if g is not None else None
        if not href:
            continue
        caption_el = fig.find(".//caption")
        caption = "".join(caption_el.itertext()).strip() if caption_el is not None else ""
        figs.append({"id": fig.get("id") or f"fig{len(figs)+1}",
                     "index": len(figs) + 1, "tif_href": href, "caption": caption})
    return figs

def _jats_dir(jats_url: str) -> str:
    p = urlparse(jats_url)
    fname = p.path.rsplit("/", 1)[-1]
    stem = fname.removesuffix(".source.xml")
    parent = p.path.rsplit("/", 1)[0].replace("/content/", "/content/biorxiv/", 1)
    return f"{p.scheme}://{p.netloc}{parent}/{stem}"

def _slug(doi: str) -> str:
    return doi.replace("/", "_")

def download_article(meta: dict, out_root: Path, variant: str = "large") -> Path:
    doi = meta["doi"]
    version = meta.get("version", "1")
    jats_url = meta["jatsxml"]
    slug = f"{_slug(doi)}_v{version}"
    dest = out_root / slug
    dest.mkdir(parents=True, exist_ok=True)
    print(f"[{slug}] XML <- {jats_url}")
    xml_bytes = polite_get(jats_url).content
    (dest / "article.xml").write_bytes(xml_bytes)
    fig_dir = _jats_dir(jats_url)
    figs = extract_figures(xml_bytes)
    print(f"[{slug}] {len(figs)} figure(s)")
    for fig in figs:
        img_url = f"{fig_dir}/F{fig['index']}.{variant}.jpg"
        img_path = dest / f"F{fig['index']}.{variant}.jpg"
        try:
            data = polite_get(img_url).content
        except httpx.HTTPStatusError as e:
            print(f"  ! F{fig['index']}: HTTP {e.response.status_code}")
            continue
        img_path.write_bytes(data)
        print(f"  F{fig['index']:<2} {len(data):>8,} B")
    return dest

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
for doi in DOIS:
    versions = fetch_by_doi(SERVER, doi)
    if not versions:
        print(f"[warn] no records for {doi}")
        continue
    records.append(max(versions, key=lambda r: int(r.get("version", 1))))

print(f"{len(records)} record(s) to download\n")
for r in records:
    download_article(r, OUT_DIR, variant=IMAGE_VARIANT)

print(f"\nDone. Saved under: {OUT_DIR.resolve()}")

## Verify download

In [ ]:
from pathlib import Path
import zipfile

# Checks the Method A download. Method B writes JATS XML and JPEG files to
# data/biorxiv_api/, not .meca archives, so this cell does not apply to it.

MECA_DIR = Path("../data/May_2026")

if not MECA_DIR.exists():
    print(f"Directory not found: {MECA_DIR.resolve()}")
else:
    meca_files = sorted(MECA_DIR.glob("*.meca"))
    print(f"Found {len(meca_files):,} .meca files in {MECA_DIR.resolve()}")
    if meca_files:
        sample = meca_files[0]
        print(f"\nSample: {sample.name}")
        with zipfile.ZipFile(sample) as zf:
            for name in zf.namelist():
                print(f"  {name}")